In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

data_dir = "/content/drive/MyDrive/Colab Notebooks/threat_6"
unsw_path = os.path.join(data_dir, "UNSW_NB15_training-set.csv")

if os.path.exists(unsw_path):
    print(f"Dataset found successfully at: {unsw_path}")
else:
    print("Dataset not found. Please check your folder path.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset found successfully at: /content/drive/MyDrive/Colab Notebooks/threat_6/UNSW_NB15_training-set.csv


In [ ]:
import pandas as pd

print("Loading UNSW-NB15 dataset for inspection...")
df_unsw = pd.read_csv(unsw_path, low_memory=False)
df_unsw.columns = df_unsw.columns.str.strip()

print(f"Dataset Shape: {df_unsw.shape}")
print(f"Columns available: {list(df_unsw.columns)[:10]} ... (Total: {len(df_unsw.columns)})")

if 'label' in df_unsw.columns:
    print("\nBinary Class Distribution (label: 0 = Normal, 1 = Attack):")
    print(df_unsw['label'].value_counts())

if 'attack_cat' in df_unsw.columns:
    print("\nAttack Categories Breakdown:")
    print(df_unsw['attack_cat'].value_counts())

Loading UNSW-NB15 dataset for inspection...
Dataset Shape: (82332, 45)
Columns available: ['id', 'dur', 'proto', 'service', 'state', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate'] ... (Total: 45)

Binary Class Distribution (label: 0 = Normal, 1 = Attack):
label
1    45332
0    37000
Name: count, dtype: int64

Attack Categories Breakdown:
attack_cat
Normal            37000
Generic           18871
Exploits          11132
Fuzzers            6062
DoS                4089
Reconnaissance     3496
Analysis            677
Backdoor            583
Shellcode           378
Worms                44
Name: count, dtype: int64


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

# 1. Clean and Prepare Features
df_clean = df_unsw.copy()
if 'id' in df_clean.columns:
    df_clean = df_clean.drop(columns=['id'])

if 'attack_cat' in df_clean.columns:
    y = df_clean['label']
    X = df_clean.drop(columns=['label', 'attack_cat'])
else:
    y = df_clean['label']
    X = df_clean.drop(columns=['label'])

# One-hot encode string/categorical columns (proto, service, state)
X = pd.get_dummies(X, drop_first=True)

# Coerce everything to numeric and drop invalid entries
for col in X.columns:
    X[col] = pd.to_numeric(X[col], errors='coerce')

X = X.replace([np.inf, -np.inf], np.nan).dropna()
y = y.loc[X.index]

print(f"Processed Feature Matrix Shape: {X.shape}")

# 2. Stratified Split
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pos_weight = (y_tr == 0).sum() / ((y_tr == 1).sum() + 1e-5)

# 3. High-Capacity XGBoost Engine
xgb_unsw = XGBClassifier(
    n_estimators=400,
    max_depth=10,
    learning_rate=0.03,
    subsample=0.85,
    colsample_bytree=0.85,
    scale_pos_weight=pos_weight * 0.3,
    random_state=42,
    n_jobs=-1
)

print("\nTraining XGBoost on UNSW-NB15...")
xgb_unsw.fit(X_tr, y_tr)

# 4. Evaluation & Threshold Optimization
y_probs = xgb_unsw.predict_proba(X_te)[:, 1]
auc_score = roc_auc_score(y_te, y_probs)
print(f"\nUNSW-NB15 Model ROC-AUC Score: {auc_score:.4f}")

print(f"\n{'Threshold':<12}{'Precision (1)':<16}{'Recall (1)':<14}{'F1 (1)':<10}{'Accuracy':<10}")
print("-" * 62)

best_f1 = 0
best_t = 0.5

for t in [0.20, 0.30, 0.40, 0.50, 0.60, 0.70]:
    preds = (y_probs >= t).astype(int)
    p = np.sum((preds == 1) & (y_te == 1)) / (np.sum(preds == 1) + 1e-10)
    r = np.sum((preds == 1) & (y_te == 1)) / (np.sum(y_te == 1) + 1e-10)
    f1 = 2 * (p * r) / (p + r + 1e-10)
    acc = accuracy_score(y_te, preds)
    if f1 > best_f1:
        best_f1 = f1
        best_t = t
    print(f"{t:<12.2f}{p:<16.4f}{r:<14.4f}{f1:<10.4f}{acc:<10.4f}")

print(f"\nOptimal Threshold: {best_t} with F1-Score: {best_f1:.4f}")

Processed Feature Matrix Shape: (82332, 187)

Training XGBoost on UNSW-NB15...

UNSW-NB15 Model ROC-AUC Score: 0.9977

Threshold   Precision (1)   Recall (1)    F1 (1)    Accuracy  
--------------------------------------------------------------
0.20        0.9843          0.9747        0.9795    0.9775    
0.30        0.9894          0.9693        0.9793    0.9774    
0.40        0.9931          0.9640        0.9783    0.9765    
0.50        0.9946          0.9576        0.9758    0.9738    
0.60        0.9957          0.9516        0.9732    0.9711    
0.70        0.9974          0.9433        0.9696    0.9675    

Optimal Threshold: 0.2 with F1-Score: 0.9795


In [ ]:
# Create a targeted Threat 06 label mask from UNSW-NB15 categories
# Threat 06 covers infiltration, reconnaissance, and unauthorized remote access (backdoors)
threat_06_categories = ['Backdoor', 'Reconnaissance', 'Exploits']

# Map target: 1 if the attack belongs to Threat 06 vectors, 0 otherwise (Normal + other attacks)
df_threat06 = df_unsw.copy()
if 'id' in df_threat06.columns:
    df_threat06 = df_threat06.drop(columns=['id'])

y_t06 = df_threat06['attack_cat'].apply(lambda x: 1 if str(x).strip() in threat_06_categories else 0)
X_t06 = df_threat06.drop(columns=['label', 'attack_cat'])

# Process features
X_t06 = pd.get_dummies(X_t06, drop_first=True)
for col in X_t06.columns:
    X_t06[col] = pd.to_numeric(X_t06[col], errors='coerce')

X_t06 = X_t06.replace([np.inf, -np.inf], np.nan).dropna()
y_t06 = y_t06.loc[X_t06.index]

print(f"Threat 06 Specific Dataset Shape: {X_t06.shape}")
print(f"Target Distribution for Threat 06:\n{y_t06.value_counts()}")

# Train the specialized Threat 06 model
X_tr_t, X_te_t, y_tr_t, y_te_t = train_test_split(
    X_t06, y_t06, test_size=0.2, random_state=42, stratify=y_t06
)

pos_weight_t = (y_tr_t == 0).sum() / ((y_tr_t == 1).sum() + 1e-5)

xgb_t06 = XGBClassifier(
    n_estimators=400,
    max_depth=10,
    learning_rate=0.03,
    subsample=0.85,
    colsample_bytree=0.85,
    scale_pos_weight=pos_weight_t * 0.3,
    random_state=42,
    n_jobs=-1
)

print("\nTraining Specialized Threat 06 Engine...")
xgb_t06.fit(X_tr_t, y_tr_t)

y_probs_t = xgb_t06.predict_proba(X_te_t)[:, 1]
print(f"\nThreat 06 Specialized ROC-AUC Score: {roc_auc_score(y_te_t, y_probs_t):.4f}")

Threat 06 Specific Dataset Shape: (82332, 187)
Target Distribution for Threat 06:
attack_cat
0    67121
1    15211
Name: count, dtype: int64

Training Specialized Threat 06 Engine...

Threat 06 Specialized ROC-AUC Score: 0.9774


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Extract feature importances from our trained Threat 06 model
importances = xgb_t06.feature_importances_
feature_names = X_t06.columns

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print("Top 10 Most Important Features Driving the Model:")
print(importance_df.head(10))

Top 10 Most Important Features Driving the Model:
             Feature  Importance
156       proto_unas    0.273356
37        ct_srv_dst    0.123584
6               sttl    0.061868
7               dttl    0.054108
184        state_INT    0.052042
28      ct_state_ttl    0.032890
170      service_dns    0.030911
38   is_sm_ips_ports    0.026427
44         proto_arp    0.018360
3             sbytes    0.015830


In [ ]:
# Drop protocol and service one-hot columns to test pure behavioral performance
pure_behavior_cols = [c for c in X_t06.columns if not c.startswith('proto_') and not c.startswith('service_')]
X_pure = X_t06[pure_behavior_cols]

print(f"Reduced Behavioral Feature Matrix Shape: {X_pure.shape}")

# Retrain with pure behavior
X_tr_p, X_te_p, y_tr_p, y_te_p = train_test_split(
    X_pure, y_t06, test_size=0.2, random_state=42, stratify=y_t06
)

pos_weight_p = (y_tr_p == 0).sum() / ((y_tr_p == 1).sum() + 1e-5)

xgb_pure = XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.03,
    subsample=0.85,
    colsample_bytree=0.85,
    scale_pos_weight=pos_weight_p * 0.3,
    random_state=42,
    n_jobs=-1
)

print("\nTraining Pure Behavioral Threat 06 Engine...")
xgb_pure.fit(X_tr_p, y_tr_p)

y_probs_p = xgb_pure.predict_proba(X_te_p)[:, 1]
print(f"\nPure Behavioral ROC-AUC Score: {roc_auc_score(y_te_p, y_probs_p):.4f}")

Reduced Behavioral Feature Matrix Shape: (82332, 45)

Training Pure Behavioral Threat 06 Engine...

Pure Behavioral ROC-AUC Score: 0.9785


In [ ]:
import pandas as pd
import numpy as np

def extract_streaming_features(df_stream):
    """
    Computes rolling-window volume metrics (Items 6 & 7) dynamically
    to complement the static behavioral features.
    """
    df = df_stream.copy()

    # Ensure a timestamp or sequential ordering column exists
    if 'timestamp' not in df.columns:
        # If no timestamp exists, simulate sequence index as time proxy
        df['timestamp'] = range(len(df))

    df = df.sort_values('timestamp')

    # Assume 'src_ip' and 'sbytes' (outbound bytes) exist in the flow log
    # If column names differ, map them to your schema
    ip_col = 'src_ip' if 'src_ip' in df.columns else 'Stime' # Fallback placeholder

    # 6. host_outbound_volume_5m: Cumulative outbound bytes for internal SrcIP
    # (Using a rolling window approach based on record count or time proxy)
    if 'sbytes' in df.columns and ip_col in df.columns:
        df['host_outbound_volume_5m'] = df.groupby(ip_col)['sbytes'].transform(
            lambda x: x.rolling(window=50, min_periods=1).sum()
        )

        # 7. host_outbound_volume_zscore: Z-score deviation against rolling mean/variance
        rolling_mean = df.groupby(ip_col)['host_outbound_volume_5m'].transform(
            lambda x: x.rolling(window=200, min_periods=5).mean()
        )
        rolling_std = df.groupby(ip_col)['host_outbound_volume_5m'].transform(
            lambda x: x.rolling(window=200, min_periods=5).std()
        ).fillna(1e-5) # Prevent division by zero

        df['host_outbound_volume_zscore'] = (df['host_outbound_volume_5m'] - rolling_mean) / rolling_std
    else:
        # Fallback dummy computation if specific columns are mapped differently
        df['host_outbound_volume_5m'] = 0.0
        df['host_outbound_volume_zscore'] = 0.0

    return df

# Example usage for testing a live batch block:
# processed_stream = extract_streaming_features(raw_incoming_df)

In [ ]:
from scapy.all import rdpcap, IP, TCP, UDP
import pandas as pd
import numpy as np
import os
import joblib

# 1. Load your saved model artifact from Google Drive
export_dir = "/content/drive/MyDrive/Colab Notebooks/threat_6/models"
model_path = os.path.join(export_dir, 'threat_06_unsw_specialized_model.pkl')

artifact = joblib.load(model_path)
model = artifact['model']
threshold = artifact['threshold']
expected_features = artifact['features']

print(f"Loaded specialized model successfully. Expecting {len(expected_features)} features.")

def pcap_to_flow_dataframe(pcap_path):
    """
    Parses a raw PCAP file and aggregates packets into basic flow records
    matching behavioral network metrics.
    """
    print(f"Reading packets from {pcap_path}...")
    packets = rdpcap(pcap_path)

    flow_dict = {}

    for pkt in packets:
        if IP in pkt:
            src_ip = pkt[IP].src
            dst_ip = pkt[IP].dst
            proto = pkt[IP].proto
            pkt_len = len(pkt)
            ttl = pkt[IP].ttl

            # Define basic 5-tuple flow key
            sport = pkt[TCP].sport if TCP in pkt else (pkt[UDP].sport if UDP in pkt else 0)
            dport = pkt[TCP].dport if TCP in pkt else (pkt[UDP].dport if UDP in pkt else 0)

            flow_key = tuple(sorted([(src_ip, sport), (dst_ip, dport)]) + [proto])

            if flow_key not in flow_dict:
                flow_dict[flow_key] = {
                    'srcip': src_ip,
                    'dstip': dst_ip,
                    'proto': str(proto),
                    'spkts': 0,
                    'dpkts': 0,
                    'sbytes': 0,
                    'dbytes': 0,
                    'sttl': ttl,
                    'dttl': 0,
                    'dur': 0.1 # Minimum placeholder duration
                }

            # Accumulate metrics based on direction
            if pkt[IP].src == src_ip:
                flow_dict[flow_key]['spkts'] += 1
                flow_dict[flow_key]['sbytes'] += pkt_len
            else:
                flow_dict[flow_key]['dpkts'] += 1
                flow_dict[flow_key]['dbytes'] += pkt_len

    df_flows = pd.DataFrame(list(flow_dict.values()))
    return df_flows

# 2. Point directly to the verified exercise PCAP file path in Google Drive
target_pcap_path = "/content/drive/MyDrive/Colab Notebooks/threat_6/2026-01-31-traffic-analysis-exercise.pcap/2026-01-31-traffic-analysis-exercise.pcap"

if os.path.exists(target_pcap_path):
    raw_flow_df = pcap_to_flow_dataframe(target_pcap_path)

    # 3. Align and predict using the loaded model
    encoded_df = pd.get_dummies(raw_flow_df, drop_first=True)
    aligned_df = encoded_df.reindex(columns=expected_features, fill_value=0)

    for col in aligned_df.columns:
        aligned_df[col] = pd.to_numeric(aligned_df[col], errors='coerce')
    aligned_df = aligned_df.fillna(0)

    probabilities = model.predict_proba(aligned_df)[:, 1]
    predictions = (probabilities >= threshold).astype(int)

    raw_flow_df['Threat_Probability'] = probabilities
    raw_flow_df['Is_Threat_06'] = predictions

    print(f"\nPCAP Processing Complete. Flagged {predictions.sum()} potential threat flows out of {len(predictions)} total flows.")
    display(raw_flow_df[['srcip', 'dstip', 'proto', 'Threat_Probability', 'Is_Threat_06']].head(10))
else:
    print(f"PCAP file not found at {target_pcap_path}.")

Loaded specialized model successfully. Expecting 187 features.
Reading packets from /content/drive/MyDrive/Colab Notebooks/threat_6/2026-01-31-traffic-analysis-exercise.pcap/2026-01-31-traffic-analysis-exercise.pcap...

PCAP Processing Complete. Flagged 156 potential threat flows out of 316 total flows.


,srcip,dstip,proto,Threat_Probability,Is_Threat_06
0,0.0.0.0,255.255.255.255,17,0.300969,1
1,10.1.21.1,255.255.255.255,17,0.299608,0
2,169.254.227.202,169.254.255.255,17,0.335636,1
3,169.254.227.202,239.255.255.250,17,0.000469,0
4,10.1.21.58,10.1.21.2,17,0.211871,0
5,10.1.21.58,224.0.0.252,17,0.000466,0
6,10.1.21.1,10.1.21.58,1,0.355631,1
7,10.1.21.58,10.1.21.2,17,0.211871,0
8,10.1.21.58,10.1.21.2,17,0.249111,0
9,10.1.21.58,10.1.21.2,17,0.223996,0


In [ ]:
# Summarize and group the flagged threats by source IP for your presentation slide
threat_summary = raw_flow_df[raw_flow_df['Is_Threat_06'] == 1].groupby('srcip').agg(
    Total_Threat_Flows=('Threat_Probability', 'count'),
    Average_Threat_Score=('Threat_Probability', 'mean'),
    Protocols=('proto', lambda x: list(set(x)))
).reset_index().sort_values(by='Total_Threat_Flows', ascending=False)

print("Top Flagged Internal/External Sources for Threat 06:")
display(threat_summary.head(10))

Top Flagged Internal/External Sources for Threat 06:


,srcip,Total_Threat_Flows,Average_Threat_Score,Protocols
3,10.1.21.58,152,0.347392,"[1, 6, 17]"
0,0.0.0.0,1,0.300969,[17]
1,10.1.21.1,1,0.355631,[1]
2,10.1.21.2,1,0.363073,[6]
4,169.254.227.202,1,0.335636,[17]


In [ ]:
import os
import joblib

# Define the export directory and ensure it exists
export_dir = "/content/drive/MyDrive/Colab Notebooks/threat_6/models"
os.makedirs(export_dir, exist_ok=True)

# Package your model, threshold, feature schema, and version metadata
updated_artifact = {
    'model': model,
    'threshold': threshold,
    'features': expected_features,
    'pipeline_version': 'threat_06_streaming_v2'
}

# Save as a new .pkl artifact
new_model_path = os.path.join(export_dir, 'threat_06_unsw_streaming_model.pkl')
joblib.dump(updated_artifact, new_model_path)

print(f"New model artifact successfully saved to: {new_model_path}")

New model artifact successfully saved to: /content/drive/MyDrive/Colab Notebooks/threat_6/models/threat_06_unsw_streaming_model.pkl
